### If necessary, install Cursor Extensions

1. From the View menu, select Extensions
2. Search for Python
3. Click on "Python" made by "ms-python" and select Install if not already installed
4. Search for Jupyter
5. Click on "Jupyter" made by "ms-toolsai" and select Install if not already installed


### Next Select the Kernel

Click on "Select Kernel" on the Top Right

Choose "Python Environments..."

Then choose the one that looks like `.venv (Python 3.12.x) .venv/bin/python` - it should be marked as "Recommended" and have a big star next to it.

Any problems with this? Head over to the troubleshooting.

### Note: you'll need to set the Kernel with every notebook..
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

# Connecting to OpenAI (or Ollama)

The next cell is where we load in the environment variables in your `.env` file and connect to OpenAI.  

If you'd like to use free Ollama instead, please see the README section "Free Alternative to Paid APIs", and if you're not sure how to do this, there's a full solution in the solutions folder (day1_with_ollama.ipynb).

## Troubleshooting if you have problems:

If you get a "Name Error" - have you run all cells from the top down? Head over to the Python Foundations guide for a bulletproof way to find and fix all Name Errors.

If that doesn't fix it, head over to the [troubleshooting](../setup/troubleshooting.ipynb) notebook for step by step code to identify the root cause and fix it!

Or, contact me! Message me or email ed@edwarddonner.com and we will get this to work.

Any concerns about API costs? See my notes in the README - costs should be minimal, and you can control it at every point. You can also use Ollama as a free alternative, which we discuss during Day 2.

In [7]:
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

In [8]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENROUTER_API_KEY')
base_url = os.getenv('BASE_URL')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook


In [9]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages

[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [10]:
#can also be openai = OpenAI() if you have the env vars set up, but let's be explicit here for clarity
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("BASE_URL")
)

response = client.chat.completions.create(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Hello AI"}]
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


## OK onwards with our first project

In [11]:
ed = fetch_website_contents("https://edwarddonner.com")
print(ed)





















Home - Edward Donner
























































 









HomeAI CurriculumProficient AI EngineerConnect FourOutsmartAn arena that pits LLMs against each other in a battle of diplomacy and deviousnessAboutPosts







Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (very amateur) and losing myself in Hacker News, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of Nebula.io. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt, acquired in 2021.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [12]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [13]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)

In [14]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is the multiple of 8?"}
]

response = client.chat.completions.create(model="gpt-4.1-nano", messages=messages)
response.choices[0].message.content

"A multiple of 8 is any number that can be divided by 8 without leaving a remainder. Examples include 8, 16, 24, 32, 40, and so on. If you have a specific number in mind, I can tell you if it's a multiple of 8!"

## And now let's build useful messages for GPT-4.1-mini, using a function

In [15]:
def message_for(website):
    return  [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]


## Time to bring it together - the API for OpenAI is very simple!

In [16]:
def create_summary_for(url):
    website = fetch_website_contents(url)
    response = client.chat.completions.create(model="gpt-4.1-nano", messages=message_for(website))
    return response.choices[0].message.content

In [17]:
create_summary_for("https://edwarddonner.com")

'# Summary of the website\n\nMeet Ed, a techie with a flair for coding, AI experiments, and amateur electronic music—basically a Renaissance person who also happens to run a website. He\'s the CTO of Nebula.io and a former startup founder who loves to ramble about Large Language Models (LLMs). He’s also an accidental Udemy hit, with courses that have somehow amassed hundreds of thousands of students worldwide. \n\n# News/Announcements\n\n- **February 17, 2026:** "Vibe Coder" turns into an "Agentic Engineer"—because who needs simple code when you can have vibes.\n- **January 4, 2026:** Build voice and virtual agents with n8n, because your AI should probably talk back.\n- **November 11, 2025:** An AI live event? Apparently, AI has its own energy now.\n- **September 15, 2025:** Launch of the AI Engineering MLOps track—because deploying AI to production is just as fun as it sounds.\n\nBasically, Ed\'s website is just a hub for his ramblings, courses, and occasional updates on his AI advent

In [18]:
# A function to display this nicely in the output, using markdown
def display_summary_for(url):
    summary = create_summary_for(url)
    display(Markdown(summary))

In [19]:
display_summary_for("https://edwarddonner.com")

# Summary of the website

Meet Ed: a coding, AI, and amateur electronic music enthusiast who loves to brag about his smash-hit Udemy courses—over 400,000 students worldwide. He’s also the co-founder of Nebula.io and a former AI startup kingpin, all while casually tossing around ideas like turning language models into “Agentic Engineers.” The site boasts a handful of recent AI-themed articles and courses, with some future-sounding projects like “Connect Four” battles between LLMs. Basically, Ed is passionate about AI, sharing knowledge, and probably annoying his friends with AI jargon at parties.

In [20]:
# Step 1: Create your prompts

system_prompt = """You're the best savage reviewer of election campaign website in the world. \
You're going to be given the contents of a website, and your job is to write a short, troll, savage review of the website.  \
Make it funny and cutting, and make sure to include at least one joke.  \
Respond in markdown, and make sure to include at least one emoji. Do not wrap the markdown in a code block - respond just with the markdown."""
user_prompt = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

# Step 2: Make the messages list

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt + fetch_website_contents("https://www.thisdaylive.com/")}
]

# Step 3: Call OpenAI
response = client.chat.completions.create(
    model="openai/gpt-3.5-turbo",
    messages=messages
)

# Step 4: print the result
print(response.choices[0].message.content)

# THISDAYLIVE – Truth and Reason 🤥

Oh look, a website that claims to be all about truth and reason, yet manages to pack in more fake news than your uncle's Facebook feed on a bad day. With headlines straight out of a Telenovela script, THISDAYLIVE is your one-stop shop for over-the-top drama and questionable reporting. 

From celebrating leading African women to seeking privatisation of NNPC, this site's content is as inconsistent as Kanye's Twitter rants. And let's not forget the breaking news about a "Fire Outbreat" at Lagos Airport - did they hire a dyslexic intern as their proofreader? 

If you're looking for a good laugh or some creative writing prompts, THISDAYLIVE has got you covered. Just don't rely on it for, you know, actual news. #FactsDontLiveHere 🙄


In [22]:
display_summary_for("https://thisdaylive.com/")

Wow, what a digital masterpiece—if your idea of a "website" is a sprawling, intentionally confusing maze of headlines that seems more like an internet scavenger hunt. 🕵️‍♂️

Summarizing this site? Easy: it’s basically the Nigerian news app that couldn’t say no to every trending headline on earth—economy, politics, health, sports—if it moves, this site’s got a story about it, even if it’s just to remind you that "dirt tricks" are back in fashion or that Nigeria is still very much "not out of the woods." 

News & announcements? Oh, plenty. CBN lowers rates, police recover firearms, and a fire at Lagos airport—because who needs content that makes sense? And the best part? The site’s dedicated to "truth and reason"—which is hilarious, because reading this is a masterclass in chaos. 

It’s like a CNN/Al Jazeera/Advertorial mashup designed by a committee with zero editing skills. Want a joke? Here's one: this website is so full of headlines, even a Nigerian politician would struggle to tell what’s real news and what’s just fluff. 

In short, visiting this site is like taking a tour through Nigeria’s current affairs—diverse, chaotic, and somehow, still more organized than the Nigerian government. Keep it up, THISDAY, your homepage is truly the "backpage" to reality. 😂